# Documentation

Two mechanisms that can achieve global glaciation we discussed in prior classes are increased silicate weathering, e.g. by having many large LIPs 
erupt in a short time, coupled with strong albedo feedback when temperatures in high latitudes drop low enough to start large glaciations, or changing continental setup for increased rainfall.
In addition, a short release of large amounts of methane might have allowed temperatures to stay above glaciation levels, allowing 
chemical weathering to continue, dropping CO2 further. Once the methane release ended, temperatures would then rapidly plummet
to glaciation levels.
While some of our group did more literature research, we attempted to implement these two mechanisms in the model.

## General considerations

Both of the neoproterozoic glaciations are relatively short, estimated to last no more than 50 million years. With default settings, each
time step of the model is 40 millions years long, so to be able to model fast changes over short timespans, we had to adjust the size
of the timesteps the model takes. We made the steps 4 million years long, that proved sufficient to show effects in our time window.

To keep things simple, we also only attempted to cause "glaciation" for one time period, not all hypothesized glaciation perdiods.

## Silicate weathering

We adjusted silicate weathering to increase during the separation of the supercontinent Rodinia from 0.75 to 0.7 Ga. 
The code for this starts at line 387 in model_functions_reformate.py. It consists of
a smooth step function going up and down, scaled relative to the silicate weathering rate
of today. A ```scale_factor``` of 3 for example means that weathing at peak is three 
times higher than it is today.

In a first attempt, we did not use smooth function, but a simple step function. This obviously (in hindsight) breaks
the differential equation solver, so we then switched to smooth functions for everything.

We used the same general exponential form that other parts of the model use for the in- and decrease of weathering rate, to effectively
model a smooth step function that goes up inside of time interval, and stays zero for the rest of time. 
The result is that within the separation we model an increase in silicate weathering flux, leading to a decrease in temperature and atmospheric CO2.
To model these in and decreases the functions need to be smoothed as the differential equations do not handle jumping values very well.

![Effect on 3x weathering rate on the climate model.](weathering_3.png )
*Figure 1: Effect on 3x weathering rate on the climate model. Vertical lines bound 
the timeframe of the glaciation event. Horizontal line is the target temperature
we attempt to reach.*

This brute for scaling of the constant weathering rate was necessary, because there is
no other way to modify weathering in the model. It is in a closed feedback loop between
CO2, temperature and weathering. Since we want to force temperature and CO2, we have 
only this lever to move.

Figure 1 above shows that we can reach the target temperature of 280K, after which 
the theory suggests that albedo effects might trigger a global glaciation by roughly
tripling weathering. Some of the literature we found suggests that the effect of breaking
up Rodinia into smaller continents could easily have caused that kind of increase in weathering, at least temporarily.

## Methane release

The overall model has an option to also simulate different levels of atmospheric methane over time.
With that option enabled, during the Archaic a model function with higher Methane concentrations is used than during the Proterozoic.

For our model, we assume a short early spike in methane release. We model this with a release of methane from 0.750 to 0.749 Ga. 
To model the release, we use the function for archean methane levels, so at least 100x that of the rest of the Proterozoic. To specify which temperature model should be used during which time, we used smoothing functions to calculate the in- and decrease. This gives us the factors w0, w1 and w2 which either equal 1 or 0, depending on which time interval the time step falls into. Multiplying these with the correct temperature calculations and adding all together lets us calculate the surface temperature displayed in the diagram.
This is done in the code from Line 333 to 346 in model_functions_reformate.py. This way the spike is only modelled if the methane option is turned on.

Modelling only the methane release without changing the silicate weathering causes a decrease in atmospheric CO2 and an increase in temperature which matches the models understanding of methane as primarily a greenhouse gas. This effect however is very small. The continental weathering flux also decreases slightly.
Adjusting the silicate weathering flux to be tree times the baseline we see that modeling with methane decreases how much the temperture drops. We have set a target temperature of 280 K where we expect the glaciaction to cover large enough parts of the world to cause a runaway albedo effect. This temperature is no longer reached with three times the baseline weathering if Methane release is modeled (Fig. 2).

We also discussed what more long term effects of methane release could be. methane in the atmosphere would allow for temepratures to stay higher while CO2 is drawn down and for weathering to continue.  Possibly enough to drop the global temperatures below the threshold for global glaciations one the methane release stops. This process however is not built into this model or the climate models which calculate the global temperatures.
Another limit of this model is that it is not currently possible to adjust how much methane is released directly.

![Effect of methane combined with 3x weathering rate on the climate model.](Methane_release_3_weathering_sclaing.png )
*Figure 2: Combining the effect of a very short burst of high methane release with 3x weathering shows that we no longer reach
the target temperature, but are able to lower CO2 to around 100ppm,
which is considered required to trigger the albedo driving glaciation.*

## Albedo

The idea behind the albedo feedback is that below a critical temperature, glaciation causes a strong negative feedback with temperature: the lower
the temperature, the higher global albedo gets, until it's close to the maximum of 1. 
In the model we used, it is assumed that this change happens very rapidly, and stops chemical weathering (almost) entirely. However, neither
albedo nor the effect of glaciation on continental weathering are implemented in the model, so it is not as straightforward to force this.

There is an option for the overall model to vary solar luminosity according to common linear curve from Gough. As a naive approach, we attempted
to simply modify the luminosity, assuming it would have the same effect as explicitly implementing albedo. The code for this
is on line 329 in model_functions_reformate.py.
In contrast to the previous step functions, the albedo functions
depends on temperature, not on time, so it can trigger whenever
a critical temperature threshold is passed. From the literature 
we chose 280K as the critical threshold.

We didn't manage to implement the full hysteresis curve by just using simple smooth functions, so we opted to just see what happens when
we implement rapid jump from 0.3 (today's global albedo) to some higher albedo (we varied the numbers) when we hit the critical temperature.

Unfortunately, this breaks the model in ways we are not able to explain. Given that the climate function using the solar luminosity to compute global temperature is a 3rd order polynomial, it is unclear what wild impacts even a relatively small change of the albedo might have. Additionally, edge effects at the boundaries of the interpolation intervals might exarcerbate the problems. 

![Forced albedo and the effect on 3x weathering rate on the climate model.](albedo_280_w.png)
*Figure 3: Stacking albedo forcing on top of he three other modifications we made, the model breaks. The temperature jumps **up** when albedo increases, not down, and stays high for a long period, even though the critical threshold is exceeded immediately.*

The model, in the function ```try_run_foward``` implements some sanity checks on model outputs. Running with all three forcings in place, these safety checks trigger and tell the user that the model yields unrealistic results and needs to re-run. However, since we are not randomizing our inputs, this behavior is deterministic.

We created the plots in Fig. 3 by disabling the safety check in
```try_run_forward``` on lines 14-16 in cell **1.3 Run model**.

# Author credit statement:
    * Xenia Schumacher: Literature Research, Presentation prep
    * Johanna Weise: Literature Research, Presentation prep
    * Dorotea Pavlovic: Literature Research, Presentation prep
    * Lea Rusterholz: Coding, Presentation prep, Documentation 
    * Jochen Wuttke: Coding, Presentation prep, Documentation
    * Gregory De Souza: consultant :)